# Clase 4 – Práctica: Optimización I, cómo aprende una red

Este cuaderno es la práctica de la Clase 4 del curso *Advanced Deep Learning* (aprendizaje profundo avanzado). Lo completas por tu cuenta, después de la clase en vivo y tras leer el capítulo "Optimización I: cómo aprende una red".

## Qué vas a practicar

- Implementas desde cero la regla de actualización del descenso de gradiente (*gradient descent*, GD), el algoritmo que ajusta los parámetros de cualquier red neuronal para reducir su pérdida.
- Visualizas cómo la tasa de aprendizaje (*learning rate*) determina si el proceso converge, converge despacio o diverge sobre una superficie de pérdida con forma de cañón estrecho.
- Construyes una función que prueba varias tasas de aprendizaje candidatas y elige automáticamente la que mejor funciona, un caso de uso habitual cuando ajustas hiperparámetros.

## Requisito previo

Antes de empezar, asegúrate de haber visto la Clase 4 ("Optimización para *machine learning*", aprendizaje automático) y de haber leído el capítulo del manual correspondiente. Necesitas entender qué es un gradiente, qué es la tasa de aprendizaje y qué es el número de condición de una función.

## Cómo trabajar con este cuaderno

Cada ejercicio tiene tres partes: el enunciado del caso, una celda de código con partes marcadas como `# TODO` que completas tú, y una solución colapsada justo debajo para que compruebes tu trabajo. **Intenta resolver cada ejercicio por tu cuenta antes de abrir la solución.**

Ejecuta primero la celda de importaciones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

## Ejercicio 1. Implementa el descenso de gradiente desde cero

### El caso

Trabajas en el equipo de ciencia de datos de una empresa que quiere entender, con un ejemplo mínimo, cómo aprende una red neuronal antes de confiar en las librerías de aprendizaje profundo (*deep learning*, DL) que automatizan este paso. Tu tarea es construir la función `descenso_de_gradiente`, la pieza central de cualquier algoritmo de optimización basado en gradientes.

Recuerda la regla de actualización del capítulo:

$$\theta_{k+1} = \theta_k - \alpha \, \nabla h(\theta_k)$$

donde $\theta_k$ es el punto en el paso $k$, $\alpha$ es la tasa de aprendizaje y $\nabla h(\theta_k)$ es el gradiente de la función objetivo evaluado en $\theta_k$. En cada paso, te mueves en dirección contraria al gradiente: la dirección de mayor crecimiento de la pérdida.

### Pistas

- La función recibe la pérdida y el gradiente como funciones de Python, así que puedes llamarlas como `gradiente(punto_actual)`.
- Guarda cada punto visitado en la lista `trayectoria`, incluido el punto inicial, para poder dibujar después el recorrido completo.
- No necesitas usar `funcion_perdida` dentro del bucle de actualización: la recibes porque la usarás en el Ejercicio 2.
- Ya tienes escrito un freno de seguridad que detiene el bucle si los valores se disparan por encima de 1e6, para evitar desbordamientos numéricos cuando la tasa de aprendizaje es demasiado alta.

In [ ]:
def descenso_de_gradiente(funcion_perdida, gradiente, punto_inicial, tasa_aprendizaje, n_pasos):
    """
    Ejecuta el descenso de gradiente y devuelve la trayectoria completa de puntos visitados.

    Parámetros
    ----------
    funcion_perdida : función que recibe un array theta y devuelve un escalar.
    gradiente : función que recibe un array theta y devuelve el gradiente (mismo tamaño que theta).
    punto_inicial : lista o array con el punto de partida theta_0.
    tasa_aprendizaje : alpha, el tamaño del paso.
    n_pasos : número de iteraciones a ejecutar como máximo.

    Devuelve
    --------
    np.ndarray de forma (num_puntos, d) con cada punto visitado, incluido el inicial.
    """
    punto_actual = np.array(punto_inicial, dtype=float)
    trayectoria = [punto_actual.copy()]

    for _ in range(n_pasos):
        # TODO: completa la regla de actualización del descenso de gradiente.
        # theta_(k+1) = theta_k - alpha * gradiente(theta_k)
        nuevo_punto = ...

        punto_actual = nuevo_punto
        trayectoria.append(punto_actual.copy())

        # Freno de seguridad: si el proceso diverge, los valores pueden desbordar
        # el tipo float. Cortamos aquí para evitar errores numéricos.
        if np.any(np.abs(punto_actual) > 1e6):
            break

    return np.array(trayectoria)

In [ ]:
# Función de pérdida cuadrática de dos parámetros: h(theta) = 0.5 * (a * theta_1^2 + b * theta_2^2)
# 'a' y 'b' son las curvaturas en cada dirección. Con b mucho mayor que a,
# la superficie se parece al cañón estrecho de la analogía del capítulo.
a, b = 1.0, 20.0  # número de condición kappa = b / a = 20

def perdida_cuadratica(theta):
    return 0.5 * (a * theta[0] ** 2 + b * theta[1] ** 2)

def gradiente_cuadratica(theta):
    return np.array([a * theta[0], b * theta[1]])

punto_inicial = [4.0, 4.0]
tasa_aprendizaje = 0.04
n_pasos = 40

trayectoria = descenso_de_gradiente(perdida_cuadratica, gradiente_cuadratica, punto_inicial, tasa_aprendizaje, n_pasos)

# Dibujamos las curvas de nivel de la función y la trayectoria seguida
theta1_rango = np.linspace(-6, 6, 200)
theta2_rango = np.linspace(-6, 6, 200)
T1, T2 = np.meshgrid(theta1_rango, theta2_rango)
Z = 0.5 * (a * T1 ** 2 + b * T2 ** 2)

fig, ax = plt.subplots(figsize=(7, 6))
ax.contour(T1, T2, Z, levels=25, cmap="viridis", alpha=0.6)
ax.plot(trayectoria[:, 0], trayectoria[:, 1], marker="o", markersize=3, color="tab:red",
        label=f"Trayectoria (alpha={tasa_aprendizaje})")
ax.plot(0, 0, "k*", markersize=15, label="Mínimo")
ax.set_xlabel("theta_1 (dirección de baja curvatura)")
ax.set_ylabel("theta_2 (dirección de alta curvatura)")
ax.set_title("Trayectoria del descenso de gradiente")
ax.legend()
plt.show()

print(f"Pérdida final: {perdida_cuadratica(trayectoria[-1]):.6f}")
print(f"Número de puntos en la trayectoria: {len(trayectoria)}")

<details>
<summary>💡 Ver solución</summary>

```python
def descenso_de_gradiente(funcion_perdida, gradiente, punto_inicial, tasa_aprendizaje, n_pasos):
    """
    Ejecuta el descenso de gradiente y devuelve la trayectoria completa de puntos visitados.
    """
    punto_actual = np.array(punto_inicial, dtype=float)
    trayectoria = [punto_actual.copy()]

    for _ in range(n_pasos):
        # Regla de actualización: nos movemos en dirección contraria al gradiente,
        # es decir, hacia donde la función decrece más rápido.
        nuevo_punto = punto_actual - tasa_aprendizaje * gradiente(punto_actual)

        punto_actual = nuevo_punto
        trayectoria.append(punto_actual.copy())

        # Freno de seguridad ante divergencia numérica.
        if np.any(np.abs(punto_actual) > 1e6):
            break

    return np.array(trayectoria)
```
</details>

## Ejercicio 2. Encuentra automáticamente una buena tasa de aprendizaje

### El caso

Un compañero de equipo entrena un modelo sobre una función de pérdida con un número de condición alto (un cañón mucho más estrecho que el del Ejercicio 1) y no sabe qué tasa de aprendizaje elegir: si la pone demasiado alta, el entrenamiento diverge; si la pone demasiado baja, tarda una eternidad. Te pide que automatices la búsqueda: que pruebes una lista de tasas candidatas y que el propio código decida cuál funciona mejor, sin que nadie tenga que mirar las gráficas una a una.

Este es un caso de uso real: antes de lanzar un entrenamiento largo y costoso, conviene explorar varias tasas de aprendizaje en pocos pasos y descartar de entrada las que divergen.

### Lo que completas

Completa la función `diverge`, que decide si una trayectoria del descenso de gradiente se ha disparado sin control. Con esa pieza, la función `buscar_mejor_tasa` (ya escrita) recorre la lista de tasas candidatas, descarta las que divergen y se queda con la que consigue la menor pérdida final.

### Pistas

- Cuando una trayectoria diverge, el último punto tiene un valor de pérdida enorme, o incluso `inf` o `NaN` (no es un número, del inglés *not a number*) si el tipo `float` se desborda.
- `np.isfinite(valor)` te dice si un valor es un número finito (`False` si es `inf` o `NaN`).
- No necesitas revisar todos los valores de la trayectoria: con mirar el último punto es suficiente, porque si el proceso diverge, la pérdida crece de forma descontrolada hacia el final.

In [ ]:
def diverge(trayectoria, funcion_perdida, umbral=1e6):
    """
    Decide si una trayectoria del descenso de gradiente ha divergido.

    Parámetros
    ----------
    trayectoria : array de puntos visitados, tal como lo devuelve descenso_de_gradiente.
    funcion_perdida : función que recibe un array theta y devuelve un escalar.
    umbral : valor de pérdida por encima del cual consideramos que el proceso se ha disparado.

    Devuelve
    --------
    True si la trayectoria ha divergido, False si se mantiene acotada.
    """
    ultimo_punto = trayectoria[-1]
    ultimo_valor = funcion_perdida(ultimo_punto)

    # TODO: decide el criterio de divergencia a partir de 'ultimo_valor' y 'umbral'.
    # Pista: una trayectoria ha divergido si su último valor de pérdida no es finito
    # (inf o NaN) o si supera el umbral.
    ha_divergido = ...

    return ha_divergido


def buscar_mejor_tasa(funcion_perdida, gradiente, punto_inicial, tasas_candidatas, n_pasos):
    """
    Prueba cada tasa de aprendizaje candidata, ejecuta el descenso de gradiente
    y devuelve la tasa que consigue la menor pérdida final sin diverger.

    Devuelve
    --------
    mejor_tasa : la tasa de aprendizaje candidata con menor pérdida final entre las que no divergen.
    resultados : diccionario {tasa: pérdida_final}, con np.inf para las tasas que divergen.
    """
    resultados = {}
    for tasa in tasas_candidatas:
        trayectoria = descenso_de_gradiente(funcion_perdida, gradiente, punto_inicial, tasa, n_pasos)
        if diverge(trayectoria, funcion_perdida):
            resultados[tasa] = np.inf
        else:
            resultados[tasa] = funcion_perdida(trayectoria[-1])

    mejor_tasa = min(resultados, key=resultados.get)
    return mejor_tasa, resultados

In [ ]:
# Función de pérdida con un cañón mucho más estrecho que en el Ejercicio 1
a2, b2 = 1.0, 100.0  # numero de condicion kappa = 100

def perdida_valle(theta):
    return 0.5 * (a2 * theta[0] ** 2 + b2 * theta[1] ** 2)

def gradiente_valle(theta):
    return np.array([a2 * theta[0], b2 * theta[1]])

punto_inicial_valle = [4.0, 4.0]
n_pasos_valle = 50
tasas_candidatas = [0.0005, 0.001, 0.005, 0.008, 0.01, 0.015, 0.02, 0.05]

mejor_tasa, resultados = buscar_mejor_tasa(
    perdida_valle, gradiente_valle, punto_inicial_valle, tasas_candidatas, n_pasos_valle
)

for tasa, valor_final in sorted(resultados.items()):
    estado = "DIVERGE" if not np.isfinite(valor_final) else f"pérdida final = {valor_final:.6f}"
    print(f"alpha = {tasa:<8} -> {estado}")

L2 = max(a2, b2)
print(f"\nMejor tasa de aprendizaje encontrada: {mejor_tasa}")
print(f"Tasa máxima teórica de estabilidad (1/L): {1 / L2:.4f}")

In [ ]:
# Visualizamos la pérdida de cada tasa candidata a lo largo de las iteraciones
fig, ax = plt.subplots(figsize=(8, 6))
for tasa in tasas_candidatas:
    trayectoria = descenso_de_gradiente(perdida_valle, gradiente_valle, punto_inicial_valle, tasa, n_pasos_valle)
    valores = [perdida_valle(theta) for theta in trayectoria]
    estilo = "--" if diverge(trayectoria, perdida_valle) else "-"
    ax.plot(valores, estilo, marker="o", markersize=2, label=f"alpha={tasa}")

ax.set_yscale("log")
ax.set_xlabel("Iteración")
ax.set_ylabel("Pérdida [escala log]")
ax.set_title(f"Comparación de tasas de aprendizaje (kappa={b2/a2:.0f})")
ax.legend(fontsize=8)
plt.show()

<details>
<summary>💡 Ver solución</summary>

```python
def diverge(trayectoria, funcion_perdida, umbral=1e6):
    """
    Decide si una trayectoria del descenso de gradiente ha divergido.
    """
    ultimo_punto = trayectoria[-1]
    ultimo_valor = funcion_perdida(ultimo_punto)

    # Ha divergido si el último valor de pérdida no es finito (inf o NaN)
    # o si supera el umbral: en ambos casos, el proceso se ha disparado sin control.
    ha_divergido = (not np.isfinite(ultimo_valor)) or (ultimo_valor > umbral)

    return ha_divergido


def buscar_mejor_tasa(funcion_perdida, gradiente, punto_inicial, tasas_candidatas, n_pasos):
    resultados = {}
    for tasa in tasas_candidatas:
        trayectoria = descenso_de_gradiente(funcion_perdida, gradiente, punto_inicial, tasa, n_pasos)
        if diverge(trayectoria, funcion_perdida):
            resultados[tasa] = np.inf
        else:
            resultados[tasa] = funcion_perdida(trayectoria[-1])

    mejor_tasa = min(resultados, key=resultados.get)
    return mejor_tasa, resultados
```
</details>

## Cierre

Acabas de construir, con NumPy puro y sin ningún framework de aprendizaje profundo, el algoritmo que hace posible que cualquier red neuronal aprenda. Cuando en la próxima clase uses `optimizer.step()` en PyTorch, sabrás exactamente qué ocurre por dentro: una resta entre el punto actual y el gradiente escalado por la tasa de aprendizaje, repetida miles de veces.

También comprobaste algo que te va a acompañar en toda tu carrera con IA: no existe una tasa de aprendizaje universal. Depende de la curvatura del problema, y automatizar su búsqueda, como en el Ejercicio 2, es una práctica habitual antes de lanzar un entrenamiento largo.

**Vuelve a tu cuaderno, cambia el número de condición (el valor de `b2`) a 500 o 1000, y comprueba cuánto más difícil se vuelve encontrar una tasa de aprendizaje que no diverja.**